# Dự đoán hủy đặt phòng — pipeline đầy đủ của mô hình tốt nhất

Luồng huấn luyện và đánh giá của champion đã chọn ở notebook 03
(**Ensemble (Top 3 theo CV): Random Forest + XGBoost + LightGBM**), chạy liền mạch từ `dataset/raw/hotel_bookings.csv` đến mô hình đã lưu:

1. Chuẩn bị dữ liệu: làm sạch, chia theo ngày đến, lưu `dataset/processed/`.
2. Cross validation theo quý 2016 và chọn ngưỡng phân loại trên dự đoán OOF.
3. Huấn luyện cuối trên toàn bộ train.
4. Đánh giá một lần trên test.
5. Lưu mô hình và `model_card.json`.

Cấu hình mô hình (`CHAMPION_SPEC`) lấy từ kết quả notebook 03; quá trình so sánh và chọn mô hình
nằm ở notebook 03. Tạo cùng các file đầu ra với notebook 02 và 03.

## 0. Cấu hình

In [1]:
import json
import platform
import time
from datetime import datetime, timezone
from importlib.metadata import version

import joblib
import numpy as np
import pandas as pd

from hotel_booking import config, data, evaluation, experiments, modeling
from hotel_booking.features import BookingFeatureBuilder

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

PROJECT_ROOT = config.find_project_root()
RAW_DATA_PATH = PROJECT_ROOT / "dataset" / "raw" / "hotel_bookings.csv"
PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
N_JOBS = -1

CHAMPION = 'Ensemble (Top 3 theo CV): Random Forest + XGBoost + LightGBM'
CHAMPION_SPEC = {'ensemble': [{'name': 'Random Forest',
                               'encoding': 'onehot',
                               'half_life_days': 45,
                               'booking_weights': False,
                               'pipeline': {},
                               'params': {'n_estimators': 300,
                                          'min_samples_leaf': 10,
                                          'max_features': 0.2,
                                          'max_depth': None,
                                          'class_weight': None}},
                              {'name': 'XGBoost',
                               'encoding': 'onehot',
                               'half_life_days': 180,
                               'booking_weights': False,
                               'pipeline': {},
                               'params': {'subsample': 0.85,
                                          'scale_pos_weight': 1.75,
                                          'reg_lambda': 1.0,
                                          'n_estimators': 400,
                                          'min_child_weight': 10,
                                          'max_depth': 4,
                                          'learning_rate': 0.03,
                                          'colsample_bytree': 0.5}},
                              {'name': 'LightGBM',
                               'encoding': 'onehot',
                               'half_life_days': 90,
                               'booking_weights': False,
                               'pipeline': {},
                               'params': {'subsample_freq': 1,
                                          'subsample': 0.7,
                                          'reg_lambda': 1.0,
                                          'num_leaves': 15,
                                          'n_estimators': 400,
                                          'min_child_samples': 100,
                                          'learning_rate': 0.02,
                                          'colsample_bytree': 1.0,
                                          'class_weight': 'balanced'}}]}
print(f"Project root: {PROJECT_ROOT}")

Project root: .

## 1. Chuẩn bị dữ liệu

In [2]:
raw_df = data.load_raw(RAW_DATA_PATH)
raw_sha256 = data.file_sha256(RAW_DATA_PATH)
clean_df, cleaning_log = data.clean_bookings(raw_df)
splits = data.time_split(clean_df)
assert splits["train"][config.TIME_COLUMN].max() < splits["test"][config.TIME_COLUMN].min()
assert splits["test"][config.TIME_COLUMN].max() < splits["production_holdout"][config.TIME_COLUMN].min()
for name, frame in splits.items():
    frame.to_csv(PROCESSED_DIR / f"{name}.csv", index=False, date_format="%Y-%m-%d")

split_summary = pd.DataFrame({
    name: {
        "rows": len(frame),
        "from": f"{frame[config.TIME_COLUMN].min():%Y-%m-%d}",
        "to": f"{frame[config.TIME_COLUMN].max():%Y-%m-%d}",
        "cancel_rate": round(float(frame[config.TARGET].mean()), 4),
    }
    for name, frame in splits.items()
}).T
input_schema = data.build_input_schema(splits["train"])
builder = BookingFeatureBuilder().fit(splits["train"])
feature_spec = {
    "dataset": "hotel_bookings.csv (Kaggle: jessemostipak/hotel-booking-demand)",
    "raw_sha256": raw_sha256,
    "target": config.TARGET,
    "id_column": config.ID_COLUMN,
    "time_column": config.TIME_COLUMN,
    "input_columns": config.INPUT_COLUMNS + [config.GROUP_SIZE_COLUMN],
    "post_booking_columns": config.POST_BOOKING_COLUMNS,
    "leakage_columns_removed": config.LEAKAGE_COLUMNS,
    "engineered_features": {"numeric": builder.numeric_features(), "categorical": builder.categorical_features()},
    "cleaning_log": cleaning_log.to_dict(orient="records"),
    "splits": {name: {"file": f"{name}.csv", **{k: (int(v) if k == "rows" else v) for k, v in row.items()}}
               for name, row in split_summary.iterrows()},
    "input_schema": input_schema,
}
(PROCESSED_DIR / "feature_spec.json").write_text(json.dumps(feature_spec, indent=2, ensure_ascii=False, default=float),
                                                 encoding="utf-8")
display(cleaning_log)
display(split_summary)

,bước,số dòng bị loại
0,Tổng số khách bằng 0,180
1,adr âm hoặc lớn hơn 1000,2
2,children hoặc babies lớn hơn 5,3


,rows,from,to,cancel_rate
train,78586,2015-07-01,2016-12-31,0.3622
test,18442,2017-01-01,2017-04-30,0.3653
production_holdout,22177,2017-05-01,2017-08-31,0.4057


## 2. Cross validation và ngưỡng phân loại

4 fold theo quý 2016 (expanding window). Ngưỡng tối đa F1 chọn trên dự đoán OOF, test chưa được mở.

In [3]:
INPUTS = feature_spec["input_columns"]
train_df = data.read_split(PROCESSED_DIR, "train")
X_train, y_train = train_df[INPUTS], train_df[config.TARGET]
cv_folds, fold_table = data.quarterly_folds(train_df[config.TIME_COLUMN])
display(fold_table)

cv_result = experiments.cross_validate_oof(modeling.build_from_spec(CHAMPION_SPEC), X_train, y_train, cv_folds,
                                           fold_table["fold"].tolist(), train_score=False)
cv_summary = cv_result.summary()
display(cv_result.folds)
print(f"ROC-AUC CV: {cv_summary['cv_roc_auc']:.4f} ± {cv_summary['cv_roc_auc_std']:.4f} "
      f"(theo lượt đặt {cv_summary['cv_roc_auc_booking']:.4f})")

has_oof = ~np.isnan(cv_result.oof)
decision_threshold, threshold_curve = evaluation.best_f1_threshold(y_train[has_oof], cv_result.oof[has_oof])
best_row = threshold_curve.loc[threshold_curve["f1"].idxmax()]
print(f"Ngưỡng: {decision_threshold:.3f} (F1 OOF {best_row['f1']:.4f}, precision {best_row['precision']:.4f}, "
      f"recall {best_row['recall']:.4f})")

,fold,train_rows,valid_from,valid_to,valid_rows
0,2016Q1,21965,2016-01-01,2016-03-31,10949
1,2016Q2,32914,2016-04-01,2016-06-30,16190
2,2016Q3,49104,2016-07-01,2016-09-30,15006
3,2016Q4,64110,2016-10-01,2016-12-31,14476


,fold,valid_rows,valid_cancel_rate,roc_auc,average_precision,roc_auc_booking,ap_booking,train_roc_auc,fit_seconds
0,2016Q1,10949,0.3076,0.8429,0.7882,0.7567,0.4788,NaN,2.3337
1,2016Q2,16190,0.3750,0.8765,0.8617,0.8269,0.6880,NaN,3.3158
2,2016Q3,15006,0.3559,0.8526,0.8104,0.8035,0.6674,NaN,4.6377
3,2016Q4,14476,0.3829,0.8607,0.8327,0.8176,0.6714,NaN,5.4450


ROC-AUC CV: 0.8582 ± 0.0123 (theo lượt đặt 0.8012)
Ngưỡng: 0.456 (F1 OOF 0.7164, precision 0.8189, recall 0.6367)


## 3. Huấn luyện cuối

Fit một luồng trên toàn bộ train để dựng lại được chính xác từ `model_spec`; dự đoán dùng mọi luồng.

In [4]:
started = time.perf_counter()
champion = modeling.set_model_threads(modeling.build_from_spec(CHAMPION_SPEC), 1).fit(X_train, y_train)
champion = modeling.set_model_threads(champion, N_JOBS)
print(f"Đã huấn luyện {CHAMPION} trên {len(X_train):,} dòng ({time.perf_counter() - started:.0f}s)")

Đã huấn luyện Ensemble (Top 3 theo CV): Random Forest + XGBoost + LightGBM trên 78,586 dòng (31s)


## 4. Đánh giá trên test (mở một lần)

In [5]:
test_df = data.read_split(PROCESSED_DIR, "test")
X_test, y_test = test_df[INPUTS], test_df[config.TARGET]
test_proba = champion.predict_proba(X_test)[:, 1]

test_metrics = evaluation.binary_metrics(y_test, test_proba, decision_threshold)
booking_metrics = evaluation.binary_metrics(y_test, test_proba, decision_threshold,
                                            sample_weight=experiments.booking_weights(X_test))
dedup_position = test_df.index.get_indexer(test_df.drop_duplicates(subset=test_df.columns.drop(config.ID_COLUMN)).index)
dedup_metrics = evaluation.binary_metrics(y_test.iloc[dedup_position], test_proba[dedup_position], decision_threshold)
test_table = pd.DataFrame({"Theo dòng (mỗi phòng)": test_metrics,
                           "Theo lượt đặt (trọng số 1 / số phòng)": booking_metrics,
                           "Bỏ dòng trùng": dedup_metrics}).T
display(test_table)
print(f"ROC-AUC: CV {cv_summary['cv_roc_auc']:.4f} → test {test_metrics['roc_auc']:.4f}")

by_hotel = evaluation.metrics_by_group(y_test, test_proba, test_df["hotel"], decision_threshold)
by_month = evaluation.metrics_by_group(y_test, test_proba, test_df[config.TIME_COLUMN].dt.to_period("M"),
                                       decision_threshold)
display(by_hotel[["group", "rows", "cancel_rate", "roc_auc", "average_precision", "f1", "precision", "recall"]])
display(by_month[["group", "rows", "cancel_rate", "roc_auc", "f1"]])

,roc_auc,average_precision,f1,precision,recall,accuracy,balanced_accuracy,brier,log_loss
Theo dòng (mỗi phòng),0.8653,0.8241,0.6890,0.7685,0.6244,0.7941,0.7581,0.1386,0.4246
Theo lượt đặt (trọng số 1 / số phòng),0.8163,0.6504,0.5626,0.6549,0.4931,0.7789,0.6939,0.1529,0.4680
Bỏ dòng trùng,0.8115,0.6406,0.5541,0.6412,0.4879,0.7784,0.6903,0.1527,0.4681


ROC-AUC: CV 0.8582 → test 0.8653


,group,rows,cancel_rate,roc_auc,average_precision,f1,precision,recall
0,City Hotel,12255,0.4252,0.8654,0.8567,0.7222,0.7774,0.6743
1,Resort Hotel,6187,0.2465,0.8467,0.7010,0.5585,0.7261,0.4538


,group,rows,cancel_rate,roc_auc,f1
0,2017-01,3673,0.3400,0.8794,0.6717
1,2017-02,4168,0.3258,0.8437,0.6340
2,2017-03,4948,0.3377,0.8613,0.6885
3,2017-04,5653,0.4348,0.8707,0.7277


## 5. Lưu mô hình

In [6]:
model_path = MODELS_DIR / "hotel_cancellation_pipeline.joblib"
joblib.dump(champion, model_path)
model_card = experiments.to_builtin({
    "model_name": CHAMPION,
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "artifact": model_path.name,
    "task": "Dự đoán is_canceled tại thời điểm đặt phòng được xác nhận",
    "input_columns": INPUTS,
    "input_schema": feature_spec["input_schema"],
    "model_spec": CHAMPION_SPEC,
    "decision_threshold": decision_threshold,
    "cv_roc_auc": cv_summary["cv_roc_auc"],
    "cv_summary": cv_summary,
    "test_metrics": test_metrics,
    "test_metrics_by_booking": booking_metrics,
    "test_metrics_deduplicated": dedup_metrics,
    "test_metrics_by_hotel": by_hotel.set_index("group")[["roc_auc", "f1", "precision", "recall"]].to_dict(orient="index"),
    "data": {"raw_sha256": feature_spec["raw_sha256"], **{name: feature_spec["splits"][name] for name in config.SPLITS}},
    "environment": {"python": platform.python_version(),
                    **{package: version(package) for package in
                       ["scikit-learn", "pandas", "numpy", "lightgbm", "xgboost", "catboost"]}},
})
card_path = MODELS_DIR / "model_card.json"
card_path.write_text(json.dumps(model_card, indent=2, ensure_ascii=False), encoding="utf-8")

reloaded = joblib.load(model_path)
assert np.allclose(reloaded.predict_proba(X_test.head(500))[:, 1], test_proba[:500])
rebuilt = modeling.set_model_threads(modeling.build_from_spec(model_card["model_spec"]), 1).fit(X_train, y_train)
assert np.abs(rebuilt.predict_proba(X_test)[:, 1] - test_proba).max() < 1e-6
print(f"Đã lưu {model_path} ({model_path.stat().st_size / 1e6:.1f} MB) và {card_path}; mô hình nạp lại và dựng lại khớp.")

Đã lưu .\models\hotel_cancellation_pipeline.joblib (70.4 MB) và .\models\model_card.json; mô hình nạp lại và dựng lại khớp.
